In [ ]:
from pathlib import Path
from typing import List, Dict, Tuple, Union
from collections import defaultdict
import json

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import mat73
from tqdm.auto import tqdm
from PIL import Image
import h5py
from scipy.stats import zscore, pearsonr

from sklearn.model_selection import train_test_split

In [ ]:
from brainscore_vision import load_dataset, load_benchmark
# from brainscore_vision.benchmark_helpers.neural_common import apply_keep_attrs

In [ ]:
from helpers import compute_ceiling_splithalf, compute_ceiling_variancebased

In [ ]:
NOISE_CEILING_THRESHOLD = 10.0

In [ ]:
def load_MH(region='IT', temporal=True):
    if temporal:
        assembly = load_dataset('MajajHong2015.temporal.public')
    else:
        assembly = load_dataset('MajajHong2015.public')
        assembly = assembly.squeeze("time_bin")


    assembly = assembly.sel(region=region)
    assembly['region'] = 'neuroid', [region] * len(assembly['neuroid'])
    assembly.load()
    assembly = assembly.transpose('presentation', 'neuroid', ...)

    # assembly: DataArray with dims ('time_bin','presentation','neuroid')
    # flatten the MultiIndex -> turn its levels into coords
    da = assembly.reset_index('presentation')  # now coords like 'repetition','image_id', 'stimulus_id', ...

    # build (stimulus, repetition) index and unstack
    da_sr = (
        da
        .set_index(presentation=[ 'stimulus_id', 'repetition'])
        .unstack('presentation')   # -> dims: time_bin, neuroid, 'stimulus_id', 'repetition'
    )

    # reorder to (neurons, time_bins, stimuli, repetitions) and export
    if 'time_bin' in da_sr.dims:
        da_n_t_s_r = da_sr.transpose('neuroid', 'time_bin', 'stimulus_id', 'repetition')
    else:
        da_n_t_s_r = da_sr.transpose('neuroid', 'stimulus_id', 'repetition')
    assembly = assembly.sortby('stimulus_id')
    da_n_t_s_r = da_n_t_s_r.sortby('stimulus_id')
    arr = da_n_t_s_r.to_numpy()  # shape: (N_neurons, N_time_bins, N_stimuli, N_reps)

    return arr, assembly, da_n_t_s_r

In [ ]:
array_V4, assembly_V4, da_n_t_s_r_V4 = load_MH('V4', temporal=False)
array_V4 = array_V4.transpose(1, 0, 2)  # (stimuli, neurons, time_bins)
array_V4.shape

In [ ]:
array_IT, assembly_IT, da_n_t_s_r_IT = load_MH('IT', temporal=False)
array_IT = array_IT.transpose(1, 0, 2)  # (stimuli, neurons, time_bins)
array_IT.shape

In [ ]:
subject_names_V4 = assembly_V4.animal.values
subject_names_IT = assembly_IT.animal.values

subject_masks = {
    'Chabo': {
        'V4': subject_names_V4 == 'Chabo',
        'IT': subject_names_IT == 'Chabo',
    },
    'Tito': {
        'V4': subject_names_V4 == 'Tito',
        'IT': subject_names_IT == 'Tito',
    },
}

SUBJECTS = ['Chabo', 'Tito']
ROIS = ['V4', 'IT']

In [ ]:
assembly_V4.attrs['stimulus_set']

In [ ]:
array_V4.shape, array_IT.shape

In [ ]:

stimulus_ids_V4 = assembly_V4.attrs['stimulus_set'].sort_values('stimulus_id').stimulus_id.values
stimulus_ids_IT = assembly_IT.attrs['stimulus_set'].sort_values('stimulus_id').stimulus_id.values
categories = assembly_V4.attrs['stimulus_set'].sort_values('stimulus_id').category_name.values

assert np.array_equal(stimulus_ids_V4, stimulus_ids_IT)
assert np.array_equal(da_n_t_s_r_V4.stimulus_id.values, stimulus_ids_V4)
assert np.array_equal(da_n_t_s_r_IT.stimulus_id.values, stimulus_ids_IT)



In [ ]:
stimulus_ids_IT.shape, assembly_IT.stimulus_id.values.shape

In [ ]:
indices = assembly_V4.attrs['stimulus_set'].index.values
indices_train, indices_test = train_test_split(indices, stratify=categories, test_size=0.1, random_state=42, shuffle=True)

stimulus_ids_train, stimulus_ids_test = stimulus_ids_V4[indices_train], stimulus_ids_V4[indices_test]


In [ ]:


noise_ceilings_variancebased = {
    sub: {
        "V4": compute_ceiling_variancebased(array_V4[:, subject_masks[sub]['V4']].transpose(1, 0, 2)),  # (neurons, stimuli, repetitions)
        "IT": compute_ceiling_variancebased(array_IT[:, subject_masks[sub]['IT']].transpose(1, 0, 2)),
    }
    for sub in SUBJECTS
}

noise_ceilings_splithalf = {
    sub: {
        "V4": compute_ceiling_splithalf(array_V4[:, subject_masks[sub]['V4']].transpose(1, 0, 2)).mean(axis=-1),
        "IT": compute_ceiling_splithalf(array_IT[:, subject_masks[sub]['IT']].transpose(1, 0, 2)).mean(axis=-1),
    }
    for sub in SUBJECTS
}


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 5), dpi=100)
bins = np.linspace(0, 100, 21)
for subject_idx, subject in enumerate(SUBJECTS):
    for i, roi in enumerate(ROIS):
        ax = axes.flatten()[subject_idx*2 + i]
        sns.histplot(noise_ceilings_variancebased[subject][roi], bins=bins, ax=ax, color='blue', label='Variance-based')
        sns.histplot(noise_ceilings_splithalf[subject][roi], bins=bins, ax=ax, color='orange', label='Split-half')
        ax.set_title(f'Noise Ceiling Distribution - {subject} {roi}')
        ax.set_xlabel('Noise Ceiling')
        ax.set_ylabel('Number of Neurons')
        ax.legend()
        ax.set_xlim(0, 100)
plt.tight_layout()

In [ ]:
valid_neuroids = { 
    subject: {
        roi: np.where(noise_ceilings_variancebased[subject][roi] >= NOISE_CEILING_THRESHOLD)[0]
        for roi in ROIS
    }
    for subject in SUBJECTS
}

In [ ]:
for subject in SUBJECTS:
    for roi in ROIS:
        num_total = noise_ceilings_variancebased[subject][roi].shape[0]
        num_valid = valid_neuroids[subject][roi].shape[0]
        print(f'Subject: {subject}, ROI: {roi}, Valid Neuroids: {num_valid}/{num_total} ({(num_valid/num_total)*100:.2f}%), minimum NC: {noise_ceilings_variancebased[subject][roi][valid_neuroids[subject][roi]].min():.2f}')

In [ ]:
noise_ceilings_variancebased_filtered = {
    subject: {
        roi: noise_ceilings_variancebased[subject][roi][valid_neuroids[subject][roi]]
        for roi in ROIS
    }
    for subject in SUBJECTS
}

noise_ceilings_splithalf_filtered = {
    subject: {
        roi: noise_ceilings_splithalf[subject][roi][valid_neuroids[subject][roi]]
        for roi in ROIS
    }
    for subject in SUBJECTS
}

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 5), dpi=100)
bins = np.linspace(0, 100, 21)
for subject_idx, subject in enumerate(SUBJECTS):
    for i, roi in enumerate(ROIS):
        ax = axes.flatten()[subject_idx*2 + i]
        sns.histplot(noise_ceilings_variancebased_filtered[subject][roi], bins=bins, ax=ax, color='blue', label='Variance-based')
        sns.histplot(noise_ceilings_splithalf_filtered[subject][roi], bins=bins, ax=ax, color='orange', label='Split-half')
        ax.set_title(f'Noise Ceiling Distribution - {subject} {roi}')
        ax.set_xlabel('Noise Ceiling')
        ax.set_ylabel('Number of Neurons')
        ax.legend()
        ax.set_xlim(0, 100)
plt.tight_layout()

In [ ]:
subject_train_data = {
    sub: {
        'V4': array_V4[indices_train][:, subject_masks[sub]['V4']][:, valid_neuroids[sub]['V4']],
        'IT': array_IT[indices_train][:, subject_masks[sub]['IT']][:, valid_neuroids[sub]['IT']],
    }
    for sub in SUBJECTS
}

subject_test_data = {
    sub: {
        'V4': array_V4[indices_test][:, subject_masks[sub]['V4']][:, valid_neuroids[sub]['V4']],
        'IT': array_IT[indices_test][:, subject_masks[sub]['IT']][:, valid_neuroids[sub]['IT']],
    }
    for sub in SUBJECTS
}

In [ ]:
for sub in SUBJECTS:
    for roi in ROIS:
        print(f"{sub} {roi} train data shape: {subject_train_data[sub][roi].shape}")
        print(f"{sub} {roi} test data shape: {subject_test_data[sub][roi].shape}")

In [ ]:
for subject in SUBJECTS:
    for region in ROIS:
        print(subject, region, noise_ceilings_variancebased[subject][region].mean(), noise_ceilings_splithalf[subject][region].mean())

In [ ]:
benchmark = load_benchmark('MajajHong2015public.IT-pls')
benchmark.ceiling

In [ ]:
benchmark = load_benchmark('MajajHong2015public.V4-pls')
benchmark.ceiling

In [ ]:
subject_train_data_avg = {
    sub: {
        roi: np.nanmean(subject_train_data[sub][roi], axis=-1)  # average over repetitions
        for roi in ROIS
    }
    for sub in SUBJECTS
}

subject_test_data_avg = {
    sub: {
        roi: np.nanmean(subject_test_data[sub][roi], axis=-1)  # average over repetitions
        for roi in ROIS
    }
    for sub in SUBJECTS
}

In [ ]:
for subject in SUBJECTS:
    for roi in ROIS:
        assert noise_ceilings_variancebased_filtered[subject][roi].shape[0] == subject_train_data[subject][roi].shape[1]
        assert noise_ceilings_variancebased_filtered[subject][roi].shape[0] == subject_test_data[subject][roi].shape[1]

### Concatenate data

In [ ]:
processed_data = {
    "train" :
        {
            "stimulus_ids": stimulus_ids_train,
            "neural_data": subject_train_data_avg,
        },
    "test" :
        {
            "stimulus_ids": stimulus_ids_test,
            "neural_data": subject_test_data_avg,
        },
    "noise_ceilings": noise_ceilings_variancebased_filtered,
}

### Metadata

In [ ]:
metadata = {
    "desc": f"""
    Majaj and Hong et al. (2015) macaque V4 and IT neural data.
    10% of the stimuli are held out as test set.
    The neural data is averaged over repetitions and over a time window of 70-170ms post stimulus onset.
    """
}
metadata_str = json.dumps(metadata, indent=2)
metadata_str = json.dumps(metadata, indent=2).encode('utf-8')

### Save to disk

In [ ]:
data_dir = '${MBS_DATA_PREP_OUTPUT_DIR}'
filename = 'bs_mh.h5'

data_dir = Path(data_dir)
data_path = data_dir / filename

In [ ]:
with h5py.File(data_path, 'w') as f:
    for split in ['train', 'test']:
        f.create_dataset(f"{split}/stimulus_ids", data=processed_data[split]['stimulus_ids'])

        for subj in tqdm(SUBJECTS):
            for roi in ROIS:
                f.create_dataset(f"{split}/neural_data/{subj}/{roi}", data=processed_data[split]['neural_data'][subj][roi])
                
    for subj in SUBJECTS:
        for roi in ROIS:
            f.create_dataset(f"noise_ceilings/{subj}/{roi}", data=processed_data['noise_ceilings'][subj][roi])

    f.attrs['metadata'] = metadata_str
    f.attrs['rois'] = ROIS
    f.attrs['subjects'] = list(SUBJECTS)
    f.attrs['splits'] = ['train', 'test']
    f.attrs['max_nc'] = 100
    f.close()


In [ ]:
loaded_data = defaultdict(dict)
with h5py.File(data_path, 'r') as f:
    splits = f.attrs['splits']
    subjects = f.attrs['subjects']
    rois = f.attrs['rois']
    for split in splits:
        loaded_data[split]['stimulus_ids'] = f[split]['stimulus_ids'][()]
        
        loaded_data[split]['neural_data'] = {}
        for subj in subjects:
            loaded_data[split]['neural_data'][subj] = {}
            for roi in rois:
                loaded_data[split]['neural_data'][subj][roi] = f[split]['neural_data'][subj][roi][()]
                
    for subj in subjects:
        loaded_data['noise_ceilings'][subj] = {}
        for roi in rois:
            loaded_data['noise_ceilings'][subj][roi] = f['noise_ceilings'][subj][roi][()]


In [ ]:
loaded_data['test'].keys()
loaded_data['test']['neural_data']['Chabo']['IT'].shape, loaded_data['noise_ceilings']['Chabo']['IT'].shape

In [ ]:

# Check the saved data
with h5py.File(data_path, 'r') as f:
    splits = f.attrs['splits']
    subjects = f.attrs['subjects']
    rois = f.attrs['rois']
    print(f.keys())
    for split in splits:
        print(f[split]['stimulus_ids'].shape)

        for subj in subjects:
            for region in rois:
                print(f[split]['neural_data'][subj][region].shape)
            
    print(json.loads(f.attrs['metadata']))